# ESA WorldCover labels

**Goal:** get per-pixel land-cover labels for the AOI, remap them into a 5-class scheme, check the class mix (this answers "is the AOI any good?"), and eyeball how well the labels line up with the imagery.

**Why this matters:** a supervised U-Net needs a *target* for every pixel. Hand-labelling satellite imagery is not feasible, so I train the model to reproduce **ESA WorldCover (10 m, 2021)** from the raw Sentinel-2 bands. WorldCover is my ground truth.

> **Key idea: only labels for *one* year are needed.** WorldCover 2021 + Sentinel-2 2021 form the labelled training pair. Once trained, the U-Net predicts land cover from imagery alone, so I can run it on 2019 and 2024 (which have no labels).

## 0. Install + authenticate

In [22]:
!pip install -q earthengine-api

In [ ]:
import ee
from IPython.display import Image as IPyImage, display

PROJECT = "landcover-riau"
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Earth Engine ready, project:", PROJECT)


def show(image, vis, region, dims=600, label=None):
    """Render an ee.Image as a static thumbnail PNG inline (see Step 2)."""
    params = dict(vis)
    params["region"] = region
    params["dimensions"] = dims
    if label:
        print(label)
    display(IPyImage(url=image.getThumbURL(params)))

## 1. AOI + the 5-class scheme

These mirror `configs/config.py` (kept inline so the notebook runs standalone on Colab).

In [ ]:
AOI_BBOX = [101.5, 0.0, 102.0, 0.5]
aoi = ee.Geometry.Rectangle(AOI_BBOX)

# 5 target classes and their map colours
CLASSES = {0: "forest", 1: "agriculture", 2: "urban", 3: "water", 4: "bare"}
CLASS_COLORS = ["#1b7837", "#e6c200", "#d7191c", "#2c7fb8", "#bdbdbd"]

## 2. Build the 2021 Sentinel-2 composite

The label year is **2021** (that's the WorldCover v200 epoch), so we pair it with a 2021 dry-season composite. Same recipe as Step 2, wrapped in a function so we can reuse it for 2019/2024 later.

In [ ]:
def s2_composite(year, region, window=("06-01", "09-30"), threshold=0.6):
    """Cloud-masked Sentinel-2 SR median composite for one dry season."""
    start, end = f"{year}-{window[0]}", f"{year}-{window[1]}"
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(region).filterDate(start, end))
    cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    s2 = (s2.linkCollection(cs, ["cs_cdf"])
            .map(lambda img: img.updateMask(img.select("cs_cdf").gte(threshold))))
    return s2.median().clip(region)


composite_2021 = s2_composite(2021, aoi)
false_color = {"bands": ["B8", "B4", "B3"], "min": 0, "max": 3000}
show(composite_2021, false_color, region=aoi, label="Sentinel-2 false colour, 2021 (vegetation = red)")

## 3. Load WorldCover and remap to our 5 classes

WorldCover has 11 native classes. We collapse them into our 5. The mapping is a judgement call — here's the reasoning:

| WorldCover (native) | code | → our class |
|---|---|---|
| Tree cover | 10 | forest |
| Mangroves | 95 | forest |
| Shrubland | 20 | agriculture |
| Grassland | 30 | agriculture |
| Cropland | 40 | agriculture |
| Herbaceous wetland | 90 | agriculture |
| Built-up | 50 | urban |
| Permanent water | 80 | water |
| Bare / sparse | 60 | bare |
| Snow/ice, Moss/lichen | 70, 100 | bare (≈0 here) |

> ⚠️ **Important caveat:** WorldCover labels **mature oil-palm plantation as "Tree cover"**, the same as natural forest. So our 'forest' class really means *tree cover* (natural + established plantation). We'll lean on **tree-cover *loss* events** (clearing) as the deforestation signal rather than trying to separate forest from plantation — that separation is a known-hard problem and a possible future extension.

In [ ]:
# WorldCover v200 = 2021 epoch. It's an ImageCollection with a single image.
wc_raw = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)

NATIVE = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
TO5    = [ 0,  1,  1,  1,  2,  4,  4,  3,  1,  0,   4]

labels = wc_raw.select("Map").remap(NATIVE, TO5).rename("class")

## 4. Look at the labels next to the imagery

If the model's targets are wrong or misaligned, nothing downstream works — so always *look*.

In [ ]:
label_vis = {"min": 0, "max": 4, "palette": CLASS_COLORS}

show(composite_2021, false_color, region=aoi, label="Sentinel-2 false colour, 2021")
show(labels, label_vis, region=aoi, label="WorldCover -> 5 classes")

print("Legend:")
for code, name in CLASSES.items():
    print(f"  {CLASS_COLORS[code]}  {code} = {name}")

## 5. Class mix — is this AOI any good?

The deferred question from Step 2. For a deforestation study we want a meaningful split between **forest** and **agriculture** (so there's actual conversion to detect), not 99% of one class.

> 📖 `frequencyHistogram` counts pixels per class server-side. We use `scale=30` (not 10) just to keep it fast — fine for percentages.

In [ ]:
hist = labels.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi, scale=30, maxPixels=1e10,
).getInfo()["class"]

total = sum(hist.values())
print("Class mix across the AOI:")
for k in sorted(hist, key=lambda x: int(x)):
    print(f"  {CLASSES[int(k)]:12s} {100 * hist[k] / total:5.1f}%")

## 6. Your turn — exercises

Use the scratch cell, then jot findings in the markdown cells below it.

1. **Class mix verdict.** Read the breakdown. Is there a healthy forest + agriculture split, or does one class dominate? This decides whether we keep the AOI or nudge the bounds.
2. **Alignment.** Compare the label map to the false-colour image. Do the green (forest) regions sit where the imagery shows dense red canopy? Spot anywhere a label looks plainly wrong.
3. **The plantation caveat.** In one sentence: given WorldCover calls mature oil palm "Tree cover", how should we interpret "forest loss" in this project? (Hint: clearing still shows up as tree-cover loss.)
4. ⭐ **Resolution.** Re-run section 5 with `scale=100` then `scale=10`. Do the class percentages move much? What does that imply about mixing scales later?

In [ ]:
# scratch — your experiments here


*1. (class mix verdict)*

*2. (alignment)*

*3. (plantation caveat)*

*4. (resolution)*